In [2]:
import pandas as pd

ratings = pd.read_csv(
    "ratings.dat",
    sep="::",
    engine="python",
    names=["UserID", "MovieID", "Rating", "Timestamp"],
    encoding="latin-1"
)

user_movie_matrix = ratings.pivot(
    index="UserID",
    columns="MovieID",
    values="Rating"
)

print(user_movie_matrix.shape)

(6040, 3706)


In [3]:
n_users = ratings["UserID"].nunique()
n_movies = ratings["MovieID"].nunique()
sparsity = 1 - len(ratings) / (n_users * n_movies)

print(f"la sparsité de la matrice est de {sparsity*100:.2f} %")


la sparsité de la matrice est de 95.53 %


## Similarité entre utilisateurs

**Cosinus** : angle entre les vecteurs de notes brutes.

$$\cos(u,v) = \frac{\sum_i r_{ui}\, r_{vi}}{\sqrt{\sum_i r_{ui}^2}\;\sqrt{\sum_i r_{vi}^2}}$$

**Pearson** : même chose, mais sur les notes centrées par la moyenne de chaque utilisateur.

$$\text{pearson}(u,v) = \frac{\sum_i (r_{ui}-\bar r_u)(r_{vi}-\bar r_v)}{\sqrt{\sum_i (r_{ui}-\bar r_u)^2}\;\sqrt{\sum_i (r_{vi}-\bar r_v)^2}}$$

Les sommes portent sur les films notés par les deux utilisateurs.

**Différence** : Pearson corrige le biais de notation (utilisateur généreux vs sévère), pas le cosinus. Pearson va de -1 à 1, le cosinus de 0 à 1 avec des notes positives.

In [4]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

#On calcule la moyenne des notes des utilisateurs pour pouvoir remplacer les colonnes vides par 0, histoire qu'il n'yait pas de probleme. 
# On va centrer les notes et donc, si une note est supérieur à 0 elle est comptée positive, si inferieure a zero negative et zero si neutre
user_means = user_movie_matrix.mean(axis=1)

# 2. Centrer les notes, puis mettre 0 pour les films non notés
R_centered = user_movie_matrix.sub(user_means, axis=0).fillna(0)

# 3. Similarité cosinus entre tous les couples d'utilisateurs
sim = cosine_similarity(R_centered)
sim_df = pd.DataFrame(sim, index=user_movie_matrix.index, columns=user_movie_matrix.index)

In [5]:
## voir les voisins d'un utilisateur
## On retire l'utilisateur de la liste pour eviter d'avoir une similarité de 1 avec lui même
u = 1  # un userId
k = 10
voisins = sim_df[u].drop(u).sort_values(ascending=False).head(k)
print(voisins)

UserID
1337    0.189242
379     0.159893
5404    0.155154
49      0.148455
2607    0.148105
5739    0.146056
5635    0.144795
2495    0.139092
4159    0.138192
4724    0.135727
Name: 1, dtype: float64


In [6]:
# Pearson entre tous les couples d'utilisateurs
## corr compare les colonnes entre elles, donc on fait .T pour avoir un utilisateur par colonne
## min_periods indique le nombre de films que doivent avoir en commmun des utilisateurs pour que l'on calcule le coeff de pearson
#sim_pearson = user_movie_matrix.T.corr(method="pearson", min_periods=5)

#### Pour prédire la note d'un utilisateur pour un film on doit:
#### 1. On garde les utilisateurs qui ont noté le film i, les autres n'apportent rien
#### 2. Parmis eux on garde les k plus similaires a u
#### 3. On calcule la moyenne pondérée de leur ecarts à leur propre moyenne et on l'ajoute a la moyenne de l'utilisateur a prédire

In [ ]:
film = 25
utilisateur = 25
noteurs = user_movie_matrix.index[user_movie_matrix[film].notna().drop(utilisateur)]
